# 第 6 课 · 失败诊断、奖励漏洞与课程验收

**本课用时约 40 分钟。** 挑战课程的假设，观察探索失败，并产出一份可供复核的实验记录。

## 学完后你能做到

- 说明增加弱项训练量为何无法解决策略零概率探索。
- 构造一个错误却会被宽松验证器奖励的例子。
- 保存完整的配置、过程数据和最终成绩。

## 跟着操作

先写下每个反事实格子的预测，再运行代码核对。

运行代码格时，把光标放在格子里，按 **Shift + Enter**。看到输出后再运行下一格。若第一次误点或修改了上方变量，选 Jupyter 菜单里的 **Kernel → Restart Kernel and Run All Cells**，从第一格重新运行。本课不要求预先懂强化学习，也不需要 GPU。

## 诊断一：训练数据再多也看不到的规则

若某指令对应的正确规则初始权重为零，按权重采样永远不会选中它。因为只有正确样本才加权，策略无法靠自己的更新跨出这个零概率状态。这是探索与反馈设计的限制，不是练习次数不足。

把第三组（「前进三格」指令）正确候选的权重设成零。比较冻结权重、原始策略、训练后的策略。

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from rsi_lab import INITIAL, NAMES, evaluate, train

broken = [row.copy() for row in INITIAL]
# 第三行是「前进三格」题，第三列正是它需要的规则。
broken[2][2] = 0
for row in broken:
    assert sum(row) > 0
weights, trace = train(seed=19, adaptive=True, initial=broken, budget=128, rounds=8)
print("前进三格正确候选权重:", broken[2][2], "->", weights[2][2])
print("这一组开发期望正确率:", evaluate(weights, range(50, 75))["by_task"][2])
assert weights[2][2] == 0

提高弱项出题比例不会帮助这个策略找到自己永远不会提出的解。系统需要另一种来源的正确答案，例如示范数据、可验证的候选生成或非零探索概率。请先为每种补救方式写下新的风险，再修改代码验证。

## 诊断二：奖励与真正目标错位

把精确正确性验证器换成只要求答案落在 0 到 9 之间的格式验证器。对题目「8 前进 3 格」，答案 9 是有效数字，却是错误答案。宽松验证器会把它保留并加强错误规则。

In [ ]:
from rsi_lab import verified

def format_only(x, task, prediction):
    return isinstance(prediction, int) and 0 <= prediction <= 9

x, task, wrong = 8, 2, 9
print("正确性验证:", verified(x, task, wrong))
print("格式验证:", format_only(x, task, wrong))
assert not verified(x, task, wrong) and format_only(x, task, wrong)

In [ ]:
for label, check in [("精确", verified), ("只查格式", format_only)]:
    result, rows = train(19, True, verifier=check)
    passed = sum(sum(row["kept"]) for row in rows)
    actual = evaluate(result, range(50, 75))["expected"]
    print(f"{label}：160 个样本中保留 {passed} 个，独立评估期望正确率 {actual:.1%}")

真实 LLM 训练中，「回答能被解析」只说明通过协议约束，不代表推理结论正确。请把这一反例改成你领域里的一个奖励漏洞，并设计一个独立可运行的验证。

## 保存并核验实验记录

记录文件包含随机种子、预算、轮数、逐轮样本数以及四种方法在多个种子上的测试成绩。最终测试集用于报告，不应拿它反复选择训练参数。

In [ ]:
import json
from pathlib import Path
from rsi_lab import INITIAL, train, evaluate

runs = []
for seed in (19, 20, 21):
    for name, adaptive, budget, rounds in [
        ("frozen", False, 0, 0),
        ("single", False, 160, 1),
        ("multi_uniform", False, 32, 5),
        ("multi_adaptive", True, 32, 5),
    ]:
        weights, trace = train(seed, adaptive, budget, rounds)
        runs.append(dict(seed=seed, mode=name, budget=budget, rounds=rounds, adaptive=adaptive, weights=weights, history=trace,
                         test=evaluate(weights, range(75, 100))))

output = Path("results")
output.mkdir(exist_ok=True)
report = dict(initial=INITIAL, runs=runs, note="Toy arithmetic experiment. Results do not establish general LLM RSI capability.")
path = output / "final_report.json"
path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
loaded = json.loads(path.read_text(encoding="utf-8"))
assert len(loaded["runs"]) == 12
assert all(len(run["history"]) == (0 if run["mode"] == "frozen" else 1 if run["mode"] == "single" else 5)
           for run in loaded["runs"])
print(path.resolve(), "包含", len(loaded["runs"]), "组完整实验")

## 用报告回答一个问题

在 Jupyter 左栏打开 results/final_report.json，找到 seed 为 19、mode 为 multi_adaptive 的记录。test.expected 是最终期望正确率，history 中的 kept 是每轮通过数。用同种子 multi_uniform 的 test.expected 作差，乘 100 得到百分点差值。种子 19 应约为 +2.5 个百分点；再算种子 21，检查方向是否一致。

## 课程毕业清单

- [ ] 能从零启动 Jupyter 并选择 Python 内核。
- [ ] 能说明策略权重、奖励校验和训练分布的作用。
- [ ] 能从同种子结果分解多轮更新与自适应采样的作用。
- [ ] 能解释零概率探索失败和格式奖励漏洞。
- [ ] 能从头运行全部课，并找到 `results/final_report.json`。
- [ ] 已保存最终报告与 Notebook。

**课程边界：** 本仓库是一套 RSI 学习实验，用明确的规则模拟「生成、筛选、更新、再生成」的迭代反馈。它不连接外部 LLM，也不代表自主改写代码、模型权重或目标函数的完整 RSI 系统。将实验迁移到 verl 前，必须另行设计正确的任务奖励、独立测试集和安全回归集。

In [ ]:
print("RSI 六课验收完成。")

[返回课程首页并选择下一节](recursive_self_improvement.ipynb)